# Video 03 - Exercise Solutions

Try the exercises in [exercises.md](../exercises.md) yourself first. These are **one** good answer each.

In [0]:
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"

In [0]:
request = [{"role": "user", "content": "List 10 tips for learning Python, one per line."}]

for limit in [20, 60, 120, 400]:
    r = client.chat.completions.create(model=MODEL, messages=request, max_tokens=limit, temperature=0)
    text = r.choices[0].message.content
    lines = [line for line in text.splitlines() if line.strip()]
    print(f"max_tokens={limit:4} | finish_reason={r.choices[0].finish_reason!r:9} | lines={len(lines):2} | tokens used={r.usage.completion_tokens}")

In [0]:
ratios = []
for target in [50, 100, 200]:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"Write a {target}-word paragraph about mountains."}],
    )
    words = len(r.choices[0].message.content.split())
    tokens = r.usage.completion_tokens
    ratios.append(words / tokens)
    print(f"asked {target:3} words | got {words:3} words | {tokens:3} tokens | words/token={words / tokens:.2f}")

print(f"\nMy rule: about {sum(ratios) / len(ratios):.2f} words per token")

In [0]:
tasks = [
    ("Extract the city where I live from: 'I live in Perth and work in Fremantle'. One word.", 0),   # facts -> low
    ("Suggest a name for a hiking club. Name only.", 1.5),                                            # creative -> high
    ("What is 2 + 2? Number only.", 0),                                                               # facts -> low
]

for prompt, temperature in tasks:
    answers = [
        client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}],
                                       temperature=temperature, max_tokens=20).choices[0].message.content.strip()
        for _ in range(3)
    ]
    print(f"T={temperature} | {prompt[:45]:45} -> {answers}")

In [0]:
class IncompleteAnswerError(Exception):
    pass

def get_reply_strict(response):
    """Return the reply text, or raise if the model was cut off."""
    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise IncompleteAnswerError("Answer was cut off by max_tokens")
    return choice.message.content

try:
    r = client.chat.completions.create(model=MODEL, messages=request, max_tokens=20)
    print(get_reply_strict(r))
except IncompleteAnswerError as e:
    print("Caught:", e)

# Prefer an error when code (e.g. an agent) will ACT on the answer: failing loudly
# is safer than silently acting on half the information.